In [1]:
import os
import pandas as pd

# Path where your files landed
data_path = r"C:\Users\gaurav\electronics_data\raw_meta_Electronics"

# Load the parquet dataset into Pandas
print("Loading Parquet shards into Pandas...")
df = pd.read_parquet(data_path)

print(f"Total raw items: {len(df):,}")
print("Columns available:", df.columns.tolist())

Loading Parquet shards into Pandas...
Total raw items: 1,610,012
Columns available: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']


In [2]:
import numpy as np
import pandas as pd

data_path = r"C:\Users\gaurav\electronics_data\raw_meta_Electronics"
df = pd.read_parquet(data_path)

# 1. Keep only essential columns to save memory
cols = [
    "parent_asin",
    "title",
    "price",
    "features",
    "average_rating",
    "rating_number",
    "main_category",
]
df = df[[c for c in cols if c in df.columns]].copy()

# 2. Clean numeric price (strip '$' and convert to float)
df["price"] = pd.to_numeric(
    df["price"].astype(str).str.replace(r"[^\d.]", "", regex=True),
    errors="coerce",
)

# 3. Filter out records missing a Title or Price
df = df.dropna(subset=["title", "price"])
df = df[df["price"] > 0]

# 4. Construct direct Amazon link from parent_asin
df["product_url"] = "https://www.amazon.com/dp/" + df["parent_asin"]

# 5. Build rich searchable text for embeddings
# Convert features list to string if it is stored as a list
if isinstance(df["features"].iloc[0], (list, np.ndarray)):
    df["features_str"] = df["features"].apply(
        lambda x: " ".join(x) if isinstance(x, (list, np.ndarray)) else ""
    )
else:
    df["features_str"] = df["features"].fillna("").astype(str)

df["search_text"] = (
    "Title: "
    + df["title"]
    + " | Specs: "
    + df["features_str"].str[:250]
    + " | Category: "
    + df["main_category"].fillna("")
)

# Save the cleaned dataset for fast subsequent loads
df[
    [
        "parent_asin",
        "title",
        "price",
        "average_rating",
        "rating_number",
        "search_text",
        "product_url",
    ]
].to_parquet("cleaned_electronics_faiss.parquet", index=False)

print(f"Cleaned dataset ready: {len(df):,} items.")
print(df[["title", "price", "product_url"]].head(3))

Cleaned dataset ready: 526,489 items.
                                               title  price  \
2  Digi-Tatoo Decal Skin Compatible With MacBook ...  19.99   
3  NotoCity Compatible with Vivoactive 4 band 22m...   9.99   
4             Motorola Droid X Essentials Combo Pack  14.99   

                            product_url  
2  https://www.amazon.com/dp/B07SM135LS  
3  https://www.amazon.com/dp/B089CNGZCW  
4  https://www.amazon.com/dp/B004E2Z88O  


In [3]:
import os
import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# Confirm GPU availability for tensor operations
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device.upper()}")

# Load the saved cleaned parquet file
df = pd.read_parquet("cleaned_electronics_faiss.parquet")
print(f"Loaded {len(df):,} cleaned electronics products.")

Using device: CUDA
Loaded 526,489 cleaned electronics products.


In [4]:
from sentence_transformers import SentenceTransformer

# Load all-MiniLM-L6-v2 directly on CUDA
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", device=device
)
print("Model loaded successfully on", device.upper())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully on CUDA


In [5]:
# Generate normalized float32 embeddings for all 526,489 items
emb = model.encode(
    df["search_text"].tolist(),
    batch_size=128,  # Adjust to 256 if your GPU VRAM permits
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/4114 [00:00<?, ?it/s]

In [6]:
import faiss

# Get embedding dimensionality (384 dimensions)
dimension = emb.shape[1]

# Create flat inner product index (for normalized vectors, IP == Cosine Similarity)
index = faiss.IndexFlatIP(dimension)

# Add embeddings to the FAISS index
index.add(emb)

print(f"Total vectors indexed: {index.ntotal:,}")

Total vectors indexed: 526,489


In [7]:
import faiss
import pandas as pd

# Persist the FAISS index
faiss.write_index(index, "electronics_faiss.index")

# Save corresponding metadata for fast retrieval mapping
df[
    [
        "parent_asin",
        "title",
        "price",
        "average_rating",
        "rating_number",
        "product_url",
    ]
].to_parquet("processed_electronics_metadata.parquet", index=False)

print("Saved 'electronics_faiss.index' and 'processed_electronics_metadata.parquet' successfully!")

Saved 'electronics_faiss.index' and 'processed_electronics_metadata.parquet' successfully!


In [8]:
def search_electronics(query_text, max_budget=None, k=5):
    # 1. Encode query and normalize vector
    query_emb = model.encode(
        [query_text], convert_to_numpy=True, normalize_embeddings=True
    )

    # 2. Retrieve extra candidates (k * 10) so budget filtering does not leave you empty
    fetch_k = min(k * 10 if max_budget is not None else k, index.ntotal)
    distances, indices = index.search(query_emb, fetch_k)

    results = []
    for score, idx in zip(distances[0], indices[0]):
        item = df.iloc[idx]
        item_price = float(item["price"])

        # 3. Check hard budget ceiling
        if max_budget is not None and item_price > max_budget:
            continue

        results.append(
            {
                "title": item["title"],
                "price": round(item_price, 2),
                "rating": item.get("average_rating", None),
                "product_url": item["product_url"],
                "similarity_score": float(score),
            }
        )

        if len(results) == k:
            break

    return results


# Example test with a strict budget cap:
query = "wireless noise cancelling over ear headphones"
budget_limit = 50.0  # in USD ($) matching dataset units

results = search_electronics(query, max_budget=budget_limit, k=5)
for i, item in enumerate(results, 1):
    print(f"{i}. {item['title']}")
    print(
        f"   Price: ${item['price']} | Rating: {item['rating']} | Score: {item['similarity_score']:.4f}"
    )
    print(f"   Buy: {item['product_url']}\n")

1. Wireless Headphones Over Ear for Active Noise Cancelling Headphones
   Price: $49.99 | Rating: 4.1 | Score: 0.7757
   Buy: https://www.amazon.com/dp/B0CBPYQ6G2

2. Qisebin E7 Active Noise Cancelling Headphones, Wireless Over Ear Bluetooth Headphones with Microphone, Deep Bass Wireless Headphones 30H Playtime, Comfortable Protein Earpads, Black
   Price: $40.22 | Rating: 4.5 | Score: 0.7219
   Buy: https://www.amazon.com/dp/B09C8JC4MK

3. Vsonus Noise Cancelling Headphones Wireless Bluetooth, Over Ear Bluetooth Headphones Noise Canceling with Microphone for Adults, 40H Playtime, Deep Bass Sound, Folding, Comfortable Earpads
   Price: $29.99 | Rating: 3.6 | Score: 0.7062
   Buy: https://www.amazon.com/dp/B0B8NYDDW6

4. Active Noise Cancelling True Wireless Earbuds Bluetooth Headphones 5.1 HiFi Stereo Headphones with ENC Microphone for iPhone Android Ear Buds with Wireless Charging Case Running Earphones (Black)
   Price: $19.99 | Rating: 4.2 | Score: 0.7056
   Buy: https://www.amazon.